In [7]:
				  
import serial
import time
import re

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Test connection
arduino = serial.Serial(port='COM3', baudrate=115200, timeout=1)
time.sleep(2)
print(f"Connected to: {arduino.port}")
print(f"Is open: {arduino.is_open}")

# training stuff
forward = False


action = None

# state space and action space definition
primary_arm_discrete_positions = 46
secondary_arm_discrete_positions = 91
total_actions = primary_arm_discrete_positions + secondary_arm_discrete_positions

# hyperparameters
discount_factor = 0.9
learn_rate = 0.001

# DQN Network
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(2, 128)
        self.fc2 = nn.Linear(128, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, total_actions)  # 137
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Initialize network and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Q = DQN().to(device)
optimizer = optim.Adam(Q.parameters(), lr=learn_rate)
criterion = nn.MSELoss()

def policy_random():
    return random.randrange(0, total_actions)

def send_command(cmd):
    arduino.write(cmd.encode() + b'\n')
    time.sleep(0.1)
    response = arduino.readline().decode().strip()
    return response

def find_best_forward_action(row, col):
    state = torch.tensor([[float(row), float(col)]], dtype=torch.float32).to(device)
    with torch.no_grad():
        q_values = Q(state)
    return torch.argmin(q_values).item()

def find_best_backward_action(row, col):
    state = torch.tensor([[float(row), float(col)]], dtype=torch.float32).to(device)
    with torch.no_grad():
        q_values = Q(state)
    return torch.argmax(q_values).item()

try:
    while True:

        if arduino.in_waiting > 0:
            line = arduino.readline().decode().strip()
			

            if line.startswith("ACTION_REQUEST"):
                action = policy_random()
                send_command(f"Action:{action}")
                print(f"Sent action: {action}")

            if line.startswith("PROCESS_EXECUTION_DATA"):
                pattern = r'(\w+):(-?\d+\.?\d*)'
                matches = re.findall(pattern, line)
                data = {key: float(value) for key, value in matches}
                
			
                state_row, state_col = int(data['state_row']), int(data['state_col'])

                action = find_best_forward_action(state_row, state_col) if forward else find_best_backward_action(state_row, state_col)
                send_command(f"GreedyAction:{action}")
                print(f"Sent greedy action: {action}")
				
                print(f"State: ({state_row}, {state_col})")
                print("---")

            if line.startswith("EXECUTION_DEBUG"):
                print("greedy action never received")

            if line.startswith("PROCESS_DATA:"):
                pattern = r'(\w+):(-?\d+\.?\d*)'
                matches = re.findall(pattern, line)
                data = {key: float(value) for key, value in matches}
                
											   
                state_row, state_col = int(data['state_row']), int(data['state_col'])
                next_state_row, next_state_col = int(data['next_state_row']), int(data['next_state_col'])
                reward = data['step_reward']
                
												   
                print(f"State: ({state_row}, {state_col})")
                print(f"Reward: {reward}")
                print(f"Next State: ({next_state_row}, {next_state_col})")
                
                # Get next action
                next_action = find_best_forward_action(next_state_row, next_state_col) if forward else find_best_backward_action(next_state_row, next_state_col)
                
                # Convert to tensors
                state = torch.tensor([[float(state_row), float(state_col)]], dtype=torch.float32).to(device)
                next_state = torch.tensor([[float(next_state_row), float(next_state_col)]], dtype=torch.float32).to(device)
                
                # Current Q-value
                q_current = Q(state)[0, action]
                
                # Target Q-value
                with torch.no_grad():
                    q_next = Q(next_state)[0, next_action]
                    q_target = reward + discount_factor * q_next
                
                # Loss and backprop
                loss = criterion(q_current, q_target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                print(f"loss: {loss.item():.4f}")

                print("---")

            if line.startswith("DEBUG"):
                print(f"Response: {line}")

		

except KeyboardInterrupt:
    print("Stopped")
    arduino.close()

Connected to: COM3
Is open: True
Sent action: 88
State: (56, 24)
Reward: -0.87
Next State: (56, 0)
loss: 14.6490
---
Sent action: 73
State: (56, 0)
Reward: 0.0
Next State: (42, 0)
loss: 153.5636
---
Sent action: 65
State: (42, 0)
Reward: 0.0
Next State: (27, 0)
loss: 3.8505
---
Sent action: 17
State: (27, 0)
Reward: 0.0
Next State: (19, 0)
loss: 6.8864
---
Sent action: 5
State: (19, 0)
Reward: 0.0
Next State: (19, 17)
loss: 21.8572
---
Sent action: 49
State: (19, 17)
Reward: 0.0
Next State: (19, 5)
loss: 77.7148
---
Sent action: 45
State: (19, 5)
Reward: 0.0
Next State: (3, 5)
loss: 2.0361
---
Sent action: 101
State: (3, 5)
Reward: 0.0
Next State: (3, 45)
loss: 52.2848
---
Sent action: 57
State: (3, 45)
Reward: 1.71
Next State: (55, 45)
loss: 706.9687
---
Sent action: 73
State: (55, 45)
Reward: 0.0
Next State: (11, 45)
loss: 11.4205
---
Sent action: 87
State: (11, 45)
Reward: 1.51
Next State: (27, 45)
loss: 259.8458
---
Sent action: 73
State: (27, 45)
Reward: 0.57
Next State: (41, 45)


IndexError: index 134 is out of bounds for dimension 1 with size 128